# INF2006 Group 12: run guide (Tasks 1–4 and the Section 6 timed runs)

Follow this top to bottom on a fresh EMR cluster. Every file, path and parameter below was checked against `main`
on 20 Nov 2026 (commit `a750b3c`).

| Part | What it gives the report |
|---|---|
| Steps 1–2, 4–5 | Setup and rerun of Tasks 1, 3 and 4, which back **Appendix A** |
| **Step 3** | The Task 2 pipeline **and the timed runs**, which produce the numbers for **Section 6** |
| Step 6 | Shutdown and ZIP checklist |

Placeholders: `<BUCKET>` = your S3 bucket, `<RUN>` = UTC run id `YYYYMMDDTHHMMSSZ`, `<CLUSTER>` = `j-XXXXXXXX`.
Commands are bash on the EMR primary node unless marked **laptop**.

# Where we stand (20 Nov)

| Task | Status | Evidence in the repo |
|---|---|---|
| 1 Corpus | Done: acquired 28–29 Oct, profiled 1 Nov | `evidence/task1/run_20261101T112713Z/`, `evidence/task1/screenshots/` |
| 2 Pipeline | First run done 9 Nov (150 GDELT dates); driver runs done 20 Nov | `README (Task 2).md`, `sql/`, `task2/`, `evidence/task2_screenshots/`, `task2_driver/` |
| 3 Model | Done: EMR runs 18 Nov, fresh-session reload passed | `notebooks/INF2006_G12_task3_*.ipynb`, `task3_saved_models/`, `evidence/task3/` |
| 4 Analytics | Done: final run `20261120T075457Z` (Q1–Q3) | `notebooks/INF2006_G12_task4_analytics.ipynb`, `evidence/task4/` |
| Section 6 timed runs | Done 20 Nov: dev ×3, full ×3 | `evidence/task2_runs/` |

**Sources used.** Task 1 stages and profiles all five sources. Tasks 2–4 use prices and GDELT only.

**Decisions to settle before the timed runs**

1. **Test boundary.** The 9 Nov data (used by Task 3's models) labels late-2017 rows from 2018 prices, giving 126,001
   test rows. The driver drops those rows by default (`USE_2018_LABELS=0`). Use `USE_2018_LABELS=1` in every timed
   run if the counts should match Sections 3 and 4.
2. **Which pipeline the report describes.** The timed runs use `task2_driver/` (Hadoop Streaming over HDFS, its own
   SQL in `task2_driver/assets/sql/`). The reported models used the 9 Nov data (local unzip on the primary node, then
   `sql/`). Section 3 and Section 6 must say which one produced which numbers.
3. **Storage.** The core nodes have 100 GiB gp3 volumes, not 200 GiB: AWS Academy rejected the 200 GiB launch
   (`docs/decisions.md` D2). Record it as a deviation in Appendix A and Section 6.
4. **Appendix letters.** Use A, B, C (AI assistance), D (contributions). The brief's checklist calls the contributions
   table "C"; note that mismatch in one line.

# Step 1: set up the cluster

1. **Laptop: get the repo.** You need Git, Python 3 and the AWS CLI.

```bash
git clone https://github.com/Hexteris/INF2006_Group12_BigData.git
cd INF2006_Group12_BigData
```

2. **Laptop: credentials and key.** Start Learner Lab, paste *AWS Details → AWS CLI: Show* into `~/.aws/credentials`,
   and download `vockey2.pem` into the repo root. On Windows, run
   `icacls vockey2.pem /inheritance:r /grant:r "${env:USERNAME}:(R)"`; on macOS, run `chmod 400 vockey2.pem`.

3. **Laptop: launch the reference cluster.** The script creates the bucket if needed, uploads `install-numpy.sh` to
   `s3://<BUCKET>/bootstrap/` and attaches it as a bootstrap action, launches with `config/instance-groups.json` and
   `config/emr-configurations.json`, saves `evidence/cluster_<CLUSTER>_<stamp>.json`, and tunnels JupyterHub
   (`https://localhost:9443`, `jovyan` / `jupyter`) and YARN (`http://localhost:8088`). Allow about 15 minutes.

```powershell
.\project-up.ps1 -Bucket <BUCKET> -KeyName vockey2 -PemFile vockey2.pem      # Windows
```
```bash
./project-up.sh -b <BUCKET>                                                    # macOS / Linux
```

If SSH times out later because your IP changed: `.\project-up.ps1 -ClusterId (Get-Content .project-cluster-id) -KeyName vockey2 -PemFile vockey2.pem`
(Windows) or `./project-up.sh -c $(cat .project-cluster-id)` (macOS / Linux).

4. **Primary node: get the code.**

```bash
sudo yum install -y git tmux
git clone https://github.com/Hexteris/INF2006_Group12_BigData.git ~/g12 && cd ~/g12
tmux new -s g12
export B=<BUCKET>
```

5. **Primary node: verify the configuration.** Expect PASS for `dfs.replication=2`, 2 live DataNodes, dynamic
   allocation off, 2 executors × 2 cores × 6g, 2g overhead, 2g driver. The record goes to
   `evidence/cluster_verify_<RUN>.txt` (and to S3 if you pass the bucket). Fix any FAIL before the timed runs.

```bash
bash scripts/cluster/verify_cluster.sh $B
```

6. **Primary node: install the notebook tooling** for the driver (not timed; note how long it took).

```bash
pip install --user notebook nbconvert bash_kernel
python3 -m bash_kernel.install --user
mkdir -p ~/task2_driver && cp task2_driver/INF2006_G12_task2_driver.ipynb task2_driver/run_matrix.sh ~/task2_driver/
ls /usr/lib/hadoop/hadoop-streaming*.jar && which hdfs hive hadoop spark-submit
```

# Step 2: stage and profile the corpus (Task 1)

The raw data is already in `s3://<BUCKET>/raw/` with its acquisition manifests. Nothing is downloaded again; each
cluster copies those exact bytes into HDFS and checks them.

1. **Stage to HDFS.** Run `notebooks/INF2006_G12_stage_corpus.ipynb` cells 1–5 in the tmux session. Each source is
   downloaded from S3, hashed, written with `dfs.replication=2`, read back and re-hashed. Expected: **1,046 files**,
   every row `match=yes`, `replication_not_2=0`.

```text
/corpus/raw/prices/v1_2013-2017/acq_2026-10-28/            1 file
/corpus/raw/sec_inventory/v1/acq_2026-10-28/               1 file
/corpus/raw/sec_edgar/v1/acq_2026-10-28/                   311 files
/corpus/raw/wikinews/20261001_meta-history/acq_2026-10-29/ 2 files
/corpus/raw/gdelt/v1_events_2015-2017/acq_2026-10-29/      731 files
```

Keep the `acq_` dates as they are. Record `download_s`, `put_s` and `readback_s` from each `staging_summary.txt`:
Section 6 reports staging time separately from the timed runs.

2. **Profile the corpus.** Give the notebook user write access, then open `notebooks/INF2006_G12_task1.ipynb` on the
   **PySpark** kernel, set `S3_BUCKET` and `FULL_RUN = True` in cell 1, and run all cells.

```bash
sudo -u hdfs hdfs dfs -mkdir -p /user/jovyan && sudo -u hdfs hdfs dfs -chown jovyan /user/jovyan
hdfs dfs -chmod -R 777 /corpus/raw/_manifests
```

Expected: prices 619,040 rows / 505 tickers; SEC 311 filings / 303 CIKs; GDELT 144,207,180 accepted lines; Wikinews
1,954,230 revisions. Results go to `s3://<BUCKET>/evidence/task1_<RUN>/`.

3. **Check.** `hdfs dfs -du -s -h /corpus/raw` should show about 26 GB including replicas. Shut down the PySpark
   kernel before Step 3: a notebook Spark session holds most of the cluster's YARN memory and the driver's jobs will wait.

# Step 3: Task 2 pipeline and the Section 6 timed runs

## 3.1 What the driver runs

`task2_driver/INF2006_G12_task2_driver.ipynb` (Bash kernel) runs the whole pipeline once, on a fresh Hive database and
fresh HDFS folders, and writes every file Section 6 needs. The clock starts at the first stage, after the raw inputs
are staged.

| Stage | Engine | What it does |
|---|---|---|
| `gdelt_streaming` | Hadoop Streaming (map-only) | reads the staged GDELT zips from HDFS, validates, projects columns, writes gzip text |
| `hive_raw_tables` | Hive | run database and external tables over staged prices and the Streaming output |
| `hive_clean_prices` | Hive | cleaned / rejected / out-of-scope price tables |
| `hive_price_splits` | Hive | chronological splits |
| `hive_clean_gdelt` | Hive | cleaned / rejected GDELT; daily features by `DATEADDED` |
| `hive_reconcile` | Hive | row-count reconciliation; any FAIL fails the run |
| `spark_features` | PySpark | rolling, market-wide and lagged GDELT features; targets; split and boundary rules |
| `spark_validate` | PySpark | re-reads the Hive tables through Spark: dates, boundaries, duplicates, leakage columns |
| `train` | Spark ML | `train_task3.py`: refits the frozen volatility and direction designs (set via `TRAIN_CMD`) |

| Profile | Prices | GDELT |
|---|---|---|
| `dev` | first 50 tickers alphabetically | first 14 archives (20150701–20150714) |
| `full` | all 505 tickers | the 150-archive project sample (`evidence/task2_runs/logs/gdelt150_paths.txt`) |

The reference configuration uses 200 Spark shuffle partitions (the Spark default) and 25 GDELT archives per map task.

## 3.2 Set the training stage

Training must be inside every timed run. `task2_driver/train_task3.py` reads this run's `model_ready_*` Hive tables
(through Spark, which also shows Hive tables are usable from Spark), fits the two frozen designs from Section 4
(median Imputer → VectorAssembler → StandardScaler → logistic regression; cutoff chosen on 2016; 2017 scored once), and
saves the PipelineModels, predictions and `train_metrics.json`. It uses the driver's 8 price features, so its metrics
are a check; Section 4's results come from the Task 3 notebooks.

```bash
export TRAIN_CMD='spark-submit --master yarn --deploy-mode client \
  --conf spark.dynamicAllocation.enabled=false \
  --num-executors 2 --executor-cores 2 --executor-memory 6g \
  --conf spark.executor.memoryOverhead=2g --driver-memory 2g \
  --conf spark.sql.shuffle.partitions=$SHUFFLE_PARTS \
  $HOME/g12/task2_driver/train_task3.py --db "$DB" --out "$OUT" --local-json "$LOCAL_RUN/train_metrics.json"'
export USE_2018_LABELS=1     # decision 1 above; keep the same value for every run
export BUCKET=$B             # each run's evidence is uploaded to s3://$B/evidence/task2_runs/
```

Keep the single quotes: `DB`, `OUT`, `LOCAL_RUN` and `SHUFFLE_PARTS` are set by the driver for each run.

## 3.3 Check disk, then do one interactive dev run

```bash
hdfs dfs -df -h /                 # note free space; a full run writes tens of GB before clean-up
cd ~/task2_driver
PROFILE=dev CONFIG=reference REP=0 jupyter nbconvert --to notebook --execute INF2006_G12_task2_driver.ipynb \
  --ExecutePreprocessor.timeout=-1 --ExecutePreprocessor.kernel_name=bash \
  --output-dir ~/task2_runs/executed_notebooks --output trial_dev
cat ~/task2_runs/dev_reference_rep0_*/run_summary.txt
```

This trial run uses `REP=0`, so it is not one of the timed repetitions. The driver has only been tested with local
stand-ins, so expect to fix something here: read `run_summary.txt` and `logs/<stage>.log`. When the trial finishes
with status OK, delete `~/task2_runs/dev_reference_rep0_*` so the aggregator ignores it. JupyterHub cannot run this
notebook: its Bash kernel is installed for the `hadoop` user on the primary node, not inside JupyterHub.

## 3.4 Run the matrix

```bash
tmux new -s task2
cd ~/task2_driver
MATRIX="dev:reference full:reference" REPS="1 2 3" RAW_GDELT_DIR=/corpus/raw/gdelt_task2_150 bash run_matrix.sh 2>&1 | tee run_matrix.log
```

That is 6 runs: dev×3 and full×3 on the reference config. Before starting, copy the 150 sample archives into their own
folder: `hdfs dfs -mkdir -p /corpus/raw/gdelt_task2_150 && hdfs dfs -cp $(cat gdelt150_paths.txt) /corpus/raw/gdelt_task2_150/`
(`gdelt150_paths.txt` is in `evidence/task2_runs/logs/`). Check `hdfs dfs -df -h /` between full runs.

Redo one run: `PROFILE=full CONFIG=reference REP=2 jupyter nbconvert --to notebook --execute INF2006_G12_task2_driver.ipynb --ExecutePreprocessor.timeout=-1 --ExecutePreprocessor.kernel_name=bash --output-dir ~/task2_runs/executed_notebooks --output run_full_reference_2`

## 3.5 Aggregate and save

`run_matrix.sh` calls `aggregate_runs.py` at the end (or run the driver's section 9). It writes
`~/task2_runs/aggregate_report.md` and `aggregate_runs.csv`.

```bash
aws s3 cp --recursive ~/task2_runs s3://$B/evidence/task2_runs/ --only-show-errors
```

## 3.6 Checks before writing Section 6

| Check | Where | Expected |
|---|---|---|
| All 9 runs finished | `run_summary.txt` per run | status OK |
| Reconciliation | `reconcile.tsv` | every row PASS; prices 619,040 = 605,887 + 23 + 13,130 |
| Same inputs within a profile | `inputs.txt` | identical across the 3 repetitions (and full/reference vs full/alt) |
| Same code | `assets_sha256.txt` | identical across all runs |
| Same outputs | fingerprint in `aggregate_report.md` | IDENTICAL within each profile/config |
| Leakage | `validation.json` | no target or future columns in model-ready tables; GDELT by `DATEADDED` + 1 day |
| Training ran | `timings.csv`, `train_metrics.json` | `train` stage present in every run |
| Settings applied | `run_manifest.json`, `evidence/cluster_verify_*.txt` | reference allocation; shuffle partitions per config |
| Memory and disk | `resources_raw.log`, `disk.csv`, `hdfs dfsadmin -report` | peak values per worker |

The model-ready counts depend on `USE_2018_LABELS`; do not copy the 9 Nov counts into Section 6. Report the driver's
counts from `features_summary.json`.

## 3.7 What goes into Section 6

Fill these from `aggregate_report.md`; every number needs its file path in Appendix B.

| Table / item | Content |
|---|---|
| Profiles | rows and bytes per profile (`inputs.txt`): dev 50 tickers / 14 archives; full 505 / 150 |
| Stage timings | per stage: median and range (min–max) of the 3 repetitions, per profile/config |
| End-to-end | total timed seconds: all 3 repetitions, median, range |
| Dev vs full | medians compared, judged against each profile's range; time per GB or per million events |
| Resources | peak memory and YARN allocation; per-DataNode disk; HDFS bytes with replicas; free space |
| Not timed | cluster start, dependency setup, S3→HDFS staging (`download_s`, `put_s`, `readback_s`) |
| Bottlenecks | slowest stage and why (from `logs/<stage>.log` and YARN); any failures, reported honestly |

No speedup is a valid finding. Keep Section 6 to about 400 words: put the full tables in Appendix B and the key ones in
the main text.

# Step 4: train and reload the models (Task 3)

1. Open JupyterHub (`https://localhost:9443`, **PySpark** kernel) and upload the three Task 3 notebooks from
   `notebooks/`. Set `BUCKET` in the first code cell of each. Inputs: `s3://<BUCKET>/task2/model_ready_v1` and
   `s3://<BUCKET>/task2/cleaned/prices` (the 9 Nov Task 2 outputs).
2. Run them **one at a time**, with Kernel → Shutdown between notebooks.

| Order | Notebook | Expected |
|---|---|---|
| 1 | `INF2006_G12_task3_volatility_labels.ipynb` | writes `task3/volatility_v1`; 349,590 / 122,801 / 123,476 rows; threshold about 0.01593 |
| 2 | `INF2006_G12_task3_direction.ipynb` | winner `logistic_full_r01`; 2017 AUC 0.513 |
| 3 | `INF2006_G12_task3_volatility.ipynb` | winner `logistic_extra`; 2017 AUC about 0.745 |

Each model notebook writes `model/`, `predictions/`, `metrics/`, `settings/` and `test_features/` to
`s3://<BUCKET>/task3final/<task>/<RUN_ID>/`.

3. **Fresh-session reload.** After Run All, Kernel → Restart, then run **only the last cell**. It loads the saved
   PipelineModel without retraining, re-scores every 2017 row and compares with the saved predictions (tolerance
   1e-9). Expected: `RELOAD CHECK PASSED`, a different Spark application ID, 0 mismatches.
4. Download each notebook (File → Download as → Notebook) so the outputs are saved in it.

Our runs: direction `20261118T150036Z`, volatility `20261118T151322Z`. Models: `task3_saved_models/<task>/<RUN_ID>/model`;
metrics, settings and reload records: `evidence/task3/<task>/<RUN_ID>/`.

# Step 5: run the analytics (Task 4)

Full guide: `docs/task4_run_guide.md`.

1. Upload `notebooks/INF2006_G12_task4_analytics.ipynb` and open it on the **PySpark** kernel. Cell 1 (`%%configure`
   and the pip installs) must finish first.
2. **Code cell 3: set `BUCKET = "s3://<BUCKET>"`** (no trailing slash). Everything else follows from it, because HDFS
   is empty on a new cluster:

| Input | Default path |
|---|---|
| Prices | `<BUCKET>/task2/cleaned/prices` |
| GDELT daily features | `<BUCKET>/task2/cleaned/gdelt/daily_features` |
| Task 2 splits (Q3) | `<BUCKET>/task3/task2/model_ready_v1` |
| Task 3 predictions (Q3) | `<BUCKET>/task3/task3final/{direction/20261118T150036Z, volatility/20261118T151322Z}/predictions` |
| Evidence upload (last cell) | `<BUCKET>/evidence/task4_<RUN_ID>/` |

   Each path can be overridden with an environment variable (`G12_PRICES`, `G12_GDELT`, `G12_T3_BASE`,
   `G12_MODEL_READY`, `G12_PRED_VOL`, `G12_PRED_DIR`). Task 3 reads prices from the same `task2/cleaned/prices` prefix.
3. Run all cells (about 1 minute; 64.5 s on our final run).

| Check | Expected |
|---|---|
| Load cell | `price rows : 605887 PASS`, `gdelt rows : 150 PASS` |
| Q1 | 595,789 scored ticker-days; 6 returns above 50% nulled |
| Q2 | 100 joined days across 3 blocks |
| Q3 | 123,476 volatility and 126,001 direction rows, both PASS; AUC 0.745 / 0.513 |
| Outputs | 13 CSV, 7 PNG and `run_manifest.json` in `evidence/task4/` |

Our final run: `20261120T075457Z`; first run archived in `evidence/task4/previous_runs/20261116T164719Z/`.

# Step 6: shut down and ship

1. **Copy everything out first.** HDFS and the primary node's disks are deleted with the cluster.

```bash
aws s3 cp --recursive ~/task2_runs s3://$B/evidence/task2_runs/ --only-show-errors
aws s3 cp --recursive ~/g12/evidence s3://$B/evidence/repo_evidence/ --only-show-errors
```

**Laptop:** `aws s3 sync s3://<BUCKET>/evidence/task2_runs/ evidence/task2_runs/`, then commit. Keep the summaries,
`timings.csv`, manifests and logs; leave out any data files.

2. **Terminate** (laptop): `.\project-down.ps1` (or `./project-down.sh`), then **End Lab**.

3. **ZIP checklist**

- [ ] `report.pdf`; notebooks for Tasks 1–4 and `task2_driver/` exported with outputs; `task3_saved_models/`; scripts and SQL
- [ ] Required headings, all appendices, ≤ 3,000 words main text, word count on the cover
- [ ] Appendix A: setup, data access, dependencies, configuration, run order with notebook filenames
- [ ] Appendix A: data versions, training cutoff, software versions, seeds (`SEED = 42`), saved-model locations
- [ ] Every verification record has check, steps/command, expected, actual, evidence location
- [ ] Every referenced file opens from the ZIP root; no credentials, raw datasets or machine-specific paths
- [ ] Section 6 reports all repetitions, medians and ranges, and failures or no-speedup results honestly
- [ ] AI assistance declared (Appendix C); contributions total 100% (Appendix D)

# Where the evidence lives

| Report section | Evidence (relative path) |
|---|---|
| 2 Corpus | `evidence/task1/run_20261101T112713Z/`, `evidence/task1/screenshots/`, `docs/corpus_inventory.md`, `docs/data_dictionary.md` |
| 3 Pipeline | `task2_driver/` and `evidence/task2_runs/<run>/` (timed runs); 9 Nov run: `sql/`, `scripts/streaming/`, `README (Task 2).md`, `evidence/task2_screenshots/E1–E6` |
| 4 Model | `notebooks/INF2006_G12_task3_*.ipynb`, `task3_saved_models/`, `evidence/task3/<task>/<RUN_ID>/` |
| 5 Analytics | `notebooks/INF2006_G12_task4_analytics.ipynb`, `evidence/task4/`, `docs/task4_run_guide.md` |
| 6 Performance | `evidence/task2_runs/aggregate_report.md`, `aggregate_runs.csv`, per-run `timings.csv`, `inputs.txt`, `resources_raw.log`, `disk.csv`, `run_manifest.json` |
| Appendix A | `project-up.ps1`, `project-up.sh`, `project-down.*`, `install-numpy.sh`, `config/*.json`, `evidence/cluster_verify_*.txt`, `evidence/**/cluster_*.json` |

When a path, parameter or script name changes, update this guide in the same commit.